In [1]:
# imports
# ==================================================
import gc
import os
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import polars as pl

## REPORTE SUBE

### Ruta base, Carga de tablas Dimencionales y archivos de las Hojas que no son Boletos

In [2]:
# Carga de parquets sin ETL, tablas dimensionales y parámetros de fecha
# ==================================================
# BASE = os.environ.get("REPORTE_BASE", r"C:\Users\usuario\OneDrive\Desktop\Reporte")
# SUBE_FOLDER = os.environ.get("REPORTE_SUBE_FOLDER", "Sube")  # aquí tu fija es 'Sube'

# SUBE_DIR = Path(BASE) / SUBE_FOLDER
# Carpeta donde están los CSV de SUBE dentro de BASE
BASE = r"C:\Users\usuario\OneDrive\Desktop\Reporte"
# PARKET_DIR = os.path.join(BASE, "Parket")
# SIN_ETL_DIR = os.path.join(PARKET_DIR, "Sin_ETL")

# Carga de tablas de dimensiones
excel_path = os.path.join(BASE, "Tablas de dimensiones transporte.xlsx")
dim_contratos_tipo = pd.read_excel(excel_path)
dim_dias_tipo = pd.read_excel(excel_path, sheet_name="Días")
dim_jurisdicciones_tipo = pd.read_excel(excel_path, sheet_name="Jurisdicciones")
dim_tarifa_tipo = pd.read_excel(excel_path, sheet_name="Tarifas")

# --------------------------------------------------
# Tablas adicionales para las pestañas del Reporte
# --------------------------------------------------

# 1) Hoja "Km por línea"  -> archivo "Km consolidado Micronauta"
km_linea_path = os.path.join(BASE, "Km consolidado Micronauta.xlsx")
df_km_por_linea = pd.read_excel(km_linea_path)

# 2) Hoja "Km entre paradas" -> archivo "Km entre paradas Micronauta"
km_paradas_path = os.path.join(BASE, "Km entre paradas Micronauta.xlsx")
df_km_entre_paradas = pd.read_excel(km_paradas_path)

# Asegurar Fecha como datetime normalizado
df_km_entre_paradas["Fecha"] = pd.to_datetime(
    df_km_entre_paradas["Fecha"], errors="coerce"
).dt.normalize()

# Columna ORIGEN = "Micronauta"
df_km_entre_paradas["ORIGEN"] = "Micronauta"

# Agregar columna "Tipo DIA" desde dimensión de días
dim_dias_simple = (
    dim_dias_tipo[["Día", "Tipo de día"]]
    .drop_duplicates("Día")
    .copy()
)
dim_dias_simple["Día"] = pd.to_datetime(dim_dias_simple["Día"]).dt.normalize()

df_km_entre_paradas = df_km_entre_paradas.merge(
    dim_dias_simple,
    left_on="Fecha",
    right_on="Día",
    how="left",
).drop(columns=["Día"])

df_km_entre_paradas = df_km_entre_paradas.rename(columns={"Tipo de día": "Tipo DIA"})

# 3) Hoja "Liquidaciones" -> archivo "liquid2"
liq_path = os.path.join(BASE, "Referencias de consolidados Sube.xlsx")

df_liquidaciones = pd.read_excel(liq_path)

# Limpiar posible BOM y espacios
df_liquidaciones.columns = [c.replace("ï»¿", "").strip() for c in df_liquidaciones.columns]

print("Columnas finales en liquid2:", df_liquidaciones.columns.tolist())

# Fecha derivada del nombre de la liquidación
# Ej: 633_20250512_124734_3_RespaldoLiquidacionProc.csv
liquid_str = df_liquidaciones["Liquidacion"].astype(str)
fecha_str8 = liquid_str.str[4:12]  # posiciones 4 a 11 -> YYYYMMDD

df_liquidaciones["Fecha"] = pd.to_datetime(
    fecha_str8,
    format="%Y%m%d",
    errors="coerce",
)

# Código empresa = primeros 3 caracteres de Liquidacion
cod_liq = liquid_str.str[:3]

mapeo_empresas_liq = {
    "633": "Coniferal",
    "634": "Tamsau",
    "636": "SiBus CBA",
    "637": "Grupo FAM Urbano",
}

df_liquidaciones["Empresa Transporte"] = cod_liq.map(mapeo_empresas_liq).fillna("Otro")

# # Parámetros de fecha (rango del reporte)
FECHA_INICIO = pd.to_datetime("01/06/2025", dayfirst=True)
FECHA_FIN    = pd.to_datetime("31/12/2026", dayfirst=True)

Columnas finales en liquid2: ['Liquidacion']


### Carga con POLARS

In [3]:
# Carga SUBE (Sube_historica) por TANDAS: no materializa las ~46M filas crudas en RAM.
# Cada tanda se agrega igual que el ETL; al final se unen los parciales.
# df_s queda vacío a propósito; el ETL usa df_s_agrupado (definido acá).
# =====================================================================

BASE = r"C:\Users\usuario\OneDrive\Desktop\Reporte"
SUBE_HIST_DIR = Path(BASE) / "Sube_historica"
# SUBE_FOLDER = os.environ.get("REPORTE_SUBE_FOLDER", "Sube")
# SUBE_HIST_DIR = Path(BASE) / SUBE_FOLDER

N_LOAD_CHUNKS = 6  # más tandas = menos RAM por paso (subí a 8–10 si tenés poca memoria)

COLS_LEER = [
    "CODIGO DE EMPRESA",
    "LINEA",
    "FECHA DE TRX",
    "TIPO DE TRX",
    "MONTO DE TRX",
    "CONTRATO",
]

GROUP_COLS = [
    "Fecha",
    "CODIGO DE EMPRESA",
    "LINEA_normalizada",
    "TIPO DE TRX",
    "MONTO DE TRX",
    "CONTRATO",
]


def _norm_linea_sube(ser: pd.Series) -> pd.Series:
    s = ser.astype(str).str.strip()
    s = s.str.replace(r"^LINEA\s+", "", regex=True).str.strip()
    return s.replace(
        {
            "B80-SIBUS": "B80",
            "LINEA600": "600",
            "A600": "600",
            "A601": "601",
            "SERVICIO ESPECIAL AEROPUERTO": "AEROBUS",
            "AEROBUS TAMSE": "AEROBUS",
            "L10": "10",
            "L11": "11",
            "L12": "12",
            "L13": "13",
            "L14": "14",
            "L15": "15",
            "L16": "16",
            "L17": "17",
            "L18": "18",
            "L19": "19",
            "L20": "20",
            "L21": "21",
            "L22": "22",
            "L23": "23",
            "L24": "24",
            "L25": "25",
            "L26": "26",
            "L27": "27",
            "L28": "28",
            "L29": "29",
            "LINEA26": "26",
            "L30 C": "30",
            "L30 SB": "30",
            "L31 SB": "31",
            "L32 SB": "32",
            "L33 SB": "33",
            "L34 C": "34",
            "L34 SB": "34",
            "L35 SB": "35",
            "L36 SB": "36",
            "L40 C": "40",
            "L41 C": "41",
            "L42 C": "42",
            "L43 C": "43",
            "L44 C": "44",
            "L45 C": "45",
            "L50": "50",
            "L51": "51",
            "L52": "52",
            "L53": "53",
            "L54": "54",
            "L60": "60",
            "L61": "61",
            "L62": "62",
            "L63": "63",
            "L64": "64",
            "L65": "65",
            "L66": "66",
            "L67": "67",
            "L68": "68",
            "L70": "70",
            "L71": "71",
            "L72": "72",
            "L73": "73",
            "L74": "74",
            "L75": "75",
            "L76": "76",
            "L80 SB": "80",
            "L81 SB": "81",
            "L82 SB": "82",
            "L83 SB": "83",
            "L84 SB": "84",
            "L85 SB": "85",
            "Barrial C2": "C2",
        }
    )


df_s = pd.DataFrame()
df_s_agrupado = pd.DataFrame()
schema_csv = None

if not SUBE_HIST_DIR.is_dir():
    print(f"Carpeta no encontrada: {SUBE_HIST_DIR}")
else:
    archivos = sorted(SUBE_HIST_DIR.glob("*.csv"))
    if not archivos:
        print(f"No se encontraron CSV en {SUBE_HIST_DIR}")
    else:
        n_arch = len(archivos)
        print(f"CSV en '{SUBE_HIST_DIR}': {n_arch} archivos — procesando en {N_LOAD_CHUNKS} tandas")
        partial_aggs: list[pd.DataFrame] = []

        for ci in range(N_LOAD_CHUNKS):
            a = ci * n_arch // N_LOAD_CHUNKS
            b = (ci + 1) * n_arch // N_LOAD_CHUNKS
            chunk_files = archivos[a:b]
            if not chunk_files:
                continue

            dfs_csv: list[pl.DataFrame] = []
            for f in chunk_files:
                if schema_csv is None:
                    df_head = pl.read_csv(f, n_rows=1, separator=";", encoding="latin1")
                    schema_csv = {c: pl.Utf8 for c in df_head.columns}
                dfs_csv.append(
                    pl.read_csv(
                        f,
                        separator=";",
                        encoding="latin1",
                        schema_overrides=schema_csv,
                    )
                )

            df_chunk = pl.concat(dfs_csv, how="diagonal")
            del dfs_csv
            gc.collect()

            have = [c for c in COLS_LEER if c in df_chunk.columns]
            df_pd = df_chunk.select(have).to_pandas()
            del df_chunk
            gc.collect()

            fechas = pd.to_datetime(
                df_pd["FECHA DE TRX"].astype(str),
                dayfirst=True,
                errors="coerce",
            )
            df_pd["Fecha"] = fechas.dt.normalize()
            if "LINEA" in df_pd.columns:
                df_pd["LINEA_normalizada"] = _norm_linea_sube(df_pd["LINEA"])
            else:
                df_pd["LINEA_normalizada"] = ""

            gcols = [c for c in GROUP_COLS if c in df_pd.columns]
            agg = (
                pl.from_pandas(df_pd[gcols])
                .group_by(gcols)
                .len()
                .rename({"len": "Conteo"})
                .to_pandas()
            )
            partial_aggs.append(agg)
            del df_pd
            gc.collect()
            print(f"  tanda {ci + 1}/{N_LOAD_CHUNKS} OK — grupos parciales: {len(agg):,}")

        df_s_agrupado = pd.concat(partial_aggs, ignore_index=True)
        df_s_agrupado = df_s_agrupado.groupby(GROUP_COLS, as_index=False)["Conteo"].sum()
        del partial_aggs
        gc.collect()
        print(f"\nFilas en df_s_agrupado (combinado): {len(df_s_agrupado):,}")
        print("df_s vacío (sin filas crudas); el ETL SUBE usa df_s_agrupado.")

CSV en 'C:\Users\usuario\OneDrive\Desktop\Reporte\Sube_historica': 788 archivos
CSV encontrado: C:\Users\usuario\OneDrive\Desktop\Reporte\Sube_historica\633_20250512_124734_3_RespaldoLiquidacionProc.csv
CSV encontrado: C:\Users\usuario\OneDrive\Desktop\Reporte\Sube_historica\633_20250513_105822_3_RespaldoLiquidacionProc.csv
CSV encontrado: C:\Users\usuario\OneDrive\Desktop\Reporte\Sube_historica\633_20250514_104946_3_RespaldoLiquidacionProc.csv
CSV encontrado: C:\Users\usuario\OneDrive\Desktop\Reporte\Sube_historica\633_20250515_111043_3_RespaldoLiquidacionProc.csv
CSV encontrado: C:\Users\usuario\OneDrive\Desktop\Reporte\Sube_historica\633_20250516_110840_3_RespaldoLiquidacionProc.csv
CSV encontrado: C:\Users\usuario\OneDrive\Desktop\Reporte\Sube_historica\633_20250519_114412_3_RespaldoLiquidacionProc.csv
CSV encontrado: C:\Users\usuario\OneDrive\Desktop\Reporte\Sube_historica\633_20250520_113512_3_RespaldoLiquidacionProc.csv
CSV encontrado: C:\Users\usuario\OneDrive\Desktop\Reporte\S

C:\Users\usuario\AppData\Local\Temp\ipykernel_5808\625939892.py:40: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  print(lf_all.schema)



Filas totales en df_s (csv, columnas clave): 46587302
Columnas en df_s: ['CODIGO DE EMPRESA', 'LINEA', 'FECHA DE TRX', 'TIPO DE TRX', 'MONTO DE TRX', 'CONTRATO', 'archivo_csv']


### ETL Sube

In [4]:
# Transformación de df_s: fecha/hora, líneas, agrupación, merges y columnas calculadas
# ===============================================================================

if "df_s_agrupado" not in globals():
    df_s_agrupado = pd.DataFrame()

if df_s.empty and df_s_agrupado.empty:
    print("df_s está vacío y no hay df_s_agrupado. Ejecutá primero la celda de carga SUBE.")
elif not df_s.empty:
    # ---------- FECHA / HORA ----------
    df_s["Fecha"] = pd.NaT
    df_s["Hora"] = ""

    n_rows = len(df_s)
    idx_limits = np.linspace(0, n_rows, 4, dtype=int)  # 0, n/3, 2n/3, n

    for i in range(3):
        start, end = idx_limits[i], idx_limits[i + 1]
        idx_slice = slice(start, end)

        serie_fechas = df_s.loc[idx_slice, "FECHA DE TRX"].astype(str)
        fechas_chunk = pd.to_datetime(
            serie_fechas,
            dayfirst=True,
            errors="coerce",
        )

        df_s.loc[idx_slice, "Fecha"] = fechas_chunk.dt.normalize()
        df_s.loc[idx_slice, "Hora"] = fechas_chunk.dt.strftime("%H:%M")

    # ---------- LÍNEA NORMALIZADA ----------
    if "LINEA" in df_s.columns:
        df_s["LINEA_normalizada"] = df_s["LINEA"].astype(str).str.strip()
        df_s["LINEA_normalizada"] = df_s["LINEA_normalizada"].str.replace(
            r"^LINEA\s+", "", regex=True
        ).str.strip()

        df_s["LINEA_normalizada"] = df_s["LINEA_normalizada"].replace({
            "B80-SIBUS": "B80",
            "LINEA600": "600",
            "A600": "600",
            "A601": "601",
            "SERVICIO ESPECIAL AEROPUERTO": "AEROBUS",
            "AEROBUS TAMSE": "AEROBUS",

            "L10": "10", "L11": "11", "L12": "12", "L13": "13", "L14": "14",
            "L15": "15", "L16": "16", "L17": "17", "L18": "18", "L19": "19",
            "L20": "20", "L21": "21", "L22": "22", "L23": "23", "L24": "24",
            "L25": "25", "L26": "26", "L27": "27", "L28": "28", "L29": "29",
            "LINEA26": "26",

            "L30 C": "30", "L30 SB": "30",
            "L31 SB": "31", "L32 SB": "32", "L33 SB": "33",
            "L34 C": "34", "L34 SB": "34", "L35 SB": "35", "L36 SB": "36",

            "L40 C": "40", "L41 C": "41", "L42 C": "42", "L43 C": "43",
            "L44 C": "44", "L45 C": "45",

            "L50": "50", "L51": "51", "L52": "52", "L53": "53", "L54": "54",

            "L60": "60", "L61": "61", "L62": "62", "L63": "63", "L64": "64",
            "L65": "65", "L66": "66", "L67": "67", "L68": "68",

            "L70": "70", "L71": "71", "L72": "72", "L73": "73", "L74": "74", "L75": "75",
            "L76": "76",

            "L80 SB": "80", "L81 SB": "81", "L82 SB": "82",
            "L83 SB": "83", "L84 SB": "84", "L85 SB": "85",

            "Barrial C2": "C2",
        })
    else:
        df_s["LINEA_normalizada"] = ""

    # ---------- AGRUPACIÓN (usando Polars para mayor velocidad) ----------
    group_cols = [
        "Fecha",
        "CODIGO DE EMPRESA",
        "LINEA_normalizada",
        "TIPO DE TRX",
        "MONTO DE TRX",
        "CONTRATO",
    ]

    # Convertimos solo las columnas necesarias a Polars y hacemos el groupby ahí
    lf_grp = pl.from_pandas(df_s[group_cols])
    df_s_agrupado = (
        lf_grp
        .group_by(group_cols)           # <-- OJO: group_by, no groupby
        .len()                          # cuenta filas por grupo
        .rename({"len": "Conteo"})      # renombrar la columna de conteo
        .to_pandas()
    )

    print("Filas en df_s_agrupado:", len(df_s_agrupado))
else:
    print(
        "ETL SUBE: usando df_s_agrupado de carga por tandas —",
        f"{len(df_s_agrupado):,}",
        "grupos únicos (menor uso de memoria).",
    )

if not df_s.empty or not df_s_agrupado.empty:
    # ---------- ETL SUBE PRINCIPAL ----------
    df_all_sube = df_s_agrupado.copy()

    df_all_sube["Fecha"] = pd.to_datetime(df_all_sube["Fecha"], errors="coerce")
    df_all_sube["Empresa Recaudadora"] = "Sube"

    # Empresa Transporte directo desde CODIGO DE EMPRESA (sin columna numérica auxiliar)
    codigo_empresa = {
        633: "Coniferal",
        634: "Tamsau",
        636: "SiBus CBA",
        637: "Grupo FAM Urbano",
    }
    cod_emp_num = pd.to_numeric(
        df_all_sube["CODIGO DE EMPRESA"].astype(str).str.extract(r"(\d+)", expand=False),
        errors="coerce"
    )
    df_all_sube["Empresa Transporte"] = cod_emp_num.map(codigo_empresa).fillna("Otro")

    # Forma de pago / Línea / Contrato
    df_all_sube["Forma Pago"] = df_all_sube["TIPO DE TRX"].astype(str)
    df_all_sube["Linea"] = df_all_sube["LINEA_normalizada"].astype(str).str.strip()
    df_all_sube["Contrato"] = (
        df_all_sube["CONTRATO"]
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
    )

    # MONTO DE TRX -> Tarifa
    serie_monto = (
        df_all_sube["MONTO DE TRX"]
        .astype(str)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
    )
    df_all_sube["Tarifa"] = pd.to_numeric(serie_monto, errors="coerce").fillna(0).round(2)

    # Conteo -> Boletos
    df_all_sube["Boletos"] = pd.to_numeric(
        df_all_sube["Conteo"], errors="coerce"
    ).fillna(0).astype("int64")

    # ---------- MERGES DIMENSIONES ----------
    dim_contratos = dim_contratos_tipo.copy()
    dim_juris = dim_jurisdicciones_tipo.copy()
    dim_dias_merge = (
        dim_dias_tipo[["Día", "Tipo de día", "Nombre de día"]]
        .drop_duplicates("Día")
        .copy()
    )
    dim_tarifas = dim_tarifa_tipo.copy()

    dim_dias_merge["Día"] = pd.to_datetime(dim_dias_merge["Día"]).dt.normalize()

    # Contratos: CONTRATO / Contrato
    col_contrato_dim = "CONTRATO" if "CONTRATO" in dim_contratos.columns else "Contrato"
    dim_contratos[col_contrato_dim] = (
        dim_contratos[col_contrato_dim]
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
    )

    # TIPO por contrato
    df_all_sube = df_all_sube.merge(
        dim_contratos[[col_contrato_dim, "TIPO"]].drop_duplicates(col_contrato_dim),
        left_on="Contrato",
        right_on=col_contrato_dim,
        how="left",
    )
    if col_contrato_dim in df_all_sube.columns:
        df_all_sube = df_all_sube.drop(columns=[col_contrato_dim])

    # Jurisdicción
    dim_juris["Tipo contrato"] = dim_juris["Tipo contrato"].astype(str).str.strip()
    df_all_sube["TIPO"] = df_all_sube["TIPO"].astype(str).str.strip()
    df_all_sube = df_all_sube.merge(
        dim_juris[["Tipo contrato", "Jurisdicción"]].drop_duplicates("Tipo contrato"),
        left_on="TIPO",
        right_on="Tipo contrato",
        how="left",
    ).drop(columns=["Tipo contrato"])

    # Tipo de día / Nombre de día
    df_all_sube = df_all_sube.merge(
        dim_dias_merge,
        left_on="Fecha",
        right_on="Día",
        how="left",
    ).drop(columns=["Día"])

    # Tarifa2 por línea (normalizando y asegurando coincidencia con dim_tarifa_tipo)
    # Limpiar nombres de columnas de la tabla de tarifas
    dim_tarifas.columns = [c.strip() for c in dim_tarifas.columns]

    col_linea_tarifa = "Línea" if "Línea" in dim_tarifas.columns else "Linea"
    # Detectar la columna Tarifa2 aunque tenga espacios o mayúsculas distintas
    col_tarifa2_dim = next(
        (c for c in dim_tarifas.columns if c.strip().upper() == "TARIFA2"),
        "Tarifa2",
    )

    dim_tarifas[col_linea_tarifa] = (
        dim_tarifas[col_linea_tarifa]
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
    )

    df_all_sube["Linea"] = (
        df_all_sube["Linea"]
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
    )

    df_all_sube = df_all_sube.merge(
        dim_tarifas[[col_linea_tarifa, col_tarifa2_dim]].drop_duplicates(col_linea_tarifa),
        left_on="Linea",
        right_on=col_linea_tarifa,
        how="left",
    ).drop(columns=[col_linea_tarifa])

    df_all_sube["Tarifa2"] = df_all_sube[col_tarifa2_dim].fillna(0)
    if col_tarifa2_dim != "Tarifa2" and col_tarifa2_dim in df_all_sube.columns:
        df_all_sube = df_all_sube.drop(columns=[col_tarifa2_dim])

    # Ajuste nocturno: Tarifa2 * 1.15
    mask_nocturno = (
        df_all_sube["Tipo de día"]
        .astype(str)
        .str.upper()
        .str.contains("NOCTURNO")
    )
    df_all_sube.loc[mask_nocturno, "Tarifa2"] = (
        df_all_sube.loc[mask_nocturno, "Tarifa2"] * 1.15
    ).round(2)

    # ---------- COLUMNAS CALCULADAS ----------
    df_all_sube["Recaudacion"] = (
        df_all_sube["Boletos"] * df_all_sube["Tarifa"]
    ).round(2)

    comision_por_contrato = (
        dim_contratos
        .drop_duplicates(col_contrato_dim)
        .set_index(col_contrato_dim)["COMISION"]
    )
    df_all_sube["COMISION"] = (
        -1 * df_all_sube["Recaudacion"] * df_all_sube["Contrato"].map(comision_por_contrato)
    ).round(2)

    df_all_sube["Reca REAL"] = np.where(
        (df_all_sube["TIPO"] == "BOS") & (df_all_sube["Empresa Recaudadora"] == "Bizland"),
        (df_all_sube["Recaudacion"] * 0.5).round(2),
        np.where(df_all_sube["TIPO"] == "BAM", 0, df_all_sube["Recaudacion"].round(2)),
    )

    df_all_sube["PERCIBIDO"] = np.where(
        df_all_sube["TIPO"].isin(["BAM", "BSC", "BEC"]),
        0,
        np.where(
            (df_all_sube["TIPO"] == "BOS") & (df_all_sube["Empresa Recaudadora"] == "Bizland"),
            (df_all_sube["Recaudacion"] * 0.5 + df_all_sube["COMISION"]).round(2),
            (df_all_sube["Recaudacion"] + df_all_sube["COMISION"]).round(2),
        ),
    )

    df_all_sube["Mes"] = df_all_sube["Fecha"].dt.month
    df_all_sube["Mes_nombre"] = df_all_sube["Fecha"].dt.month_name("es_ES")

    # ---------- VALORIZACION ----------
    forma_pago_integ = df_all_sube["Forma Pago"].isin([
        "Uso Transporte con Integracion",
        "Uso Tarjeta Digital con Integración",
        "Uso tarjeta digital QR con integración",
    ])

    cond_tarifa2 = (
        (df_all_sube["Jurisdicción"] == "MUNICIPIO") |
        (df_all_sube["TIPO"].isin(["BEC", "BSC", "BAM"]))
    )

    emp_rec_upper = df_all_sube["Empresa Recaudadora"].astype(str).str.upper()

    cond_atributos = (df_all_sube["TIPO"] == "ATRIBUTOS")
    cond_bos_micronauta = (df_all_sube["TIPO"] == "BOS") & (emp_rec_upper == "MICRONAUTA")
    cond_bos_sube = (df_all_sube["TIPO"] == "BOS") & (emp_rec_upper == "SUBE")

    valor_unitario = np.select(
        [
            forma_pago_integ,
            cond_tarifa2,
            cond_atributos,
            cond_bos_micronauta,
            cond_bos_sube,
        ],
        [
            0,
            df_all_sube["Tarifa2"],
            df_all_sube["Tarifa"] / 0.45 / 1.105 * 0.55,
            df_all_sube["Tarifa"] * 0.5,
            df_all_sube["Tarifa"],
        ],
        default=0,
    )

    df_all_sube["VALORIZACION"] = (valor_unitario * df_all_sube["Boletos"]).round(2)

    # ---------- LIMPIEZA DE COLUMNAS TÉCNICAS ----------
    cols_sobra = [
        "CODIGO DE EMPRESA",
        "LINEA_normalizada",
        "TIPO DE TRX",
        "MONTO DE TRX",
        "Conteo",
        "CONTRATO_x",
        "CONTRATO_y",
        # si no querés ver estas en el detalle, también podés agregarlas acá:
        # "Empresa Recaudadora", "Forma Pago", "Linea", "Contrato", "Tarifa2", "Tarifa",
    ]
    df_all_sube = df_all_sube.drop(columns=[c for c in cols_sobra if c in df_all_sube.columns])

    print("Filas en df_all_sube (SUBE con dimensiones y columnas calculadas):", len(df_all_sube))
    df_all_sube.head()

Filas en df_s_agrupado: 264041
Filas en df_all_sube (SUBE con dimensiones y columnas calculadas): 264041


### Carga y ETL Micronauta

In [5]:
# Integrar boletos Micronauta (query micronauta.csv) al mismo esquema que df_all_sube
# ======================================================================

micro_path = os.path.join(BASE, "query micronauta.csv")

# Leer CSV Micronauta
df_mic_raw = pd.read_csv(
    micro_path,
    sep=";",          # deja ";" porque tu archivo actual lo usa
    encoding="latin1",
)

# Limpiar BOM, comillas y espacios en los encabezados
df_mic_raw.columns = [
    c.replace("ï»¿", "").replace('"', "").strip()
    for c in df_mic_raw.columns
]

print("Columnas en query micronauta (limpias):", df_mic_raw.columns.tolist())

# Si viene transad_id, lo ignoramos (no lo usamos en el reporte)
if "transad_id" in df_mic_raw.columns:
    df_mic_raw = df_mic_raw.drop(columns=["transad_id"])

# --------------------------------------------------------------------
# 1) Fecha y hora: separar FECHA (date) y HORA (HH:MM) usando formato real YYYY-MM-DD HH:MM:SS
# --------------------------------------------------------------------
dt_mic = pd.to_datetime(
    df_mic_raw["Fecha"],
    errors="coerce",          # por si hay algún valor raro
)  # Pandas reconoce bien '2026-01-01 10:19:44'

df_mic_base = pd.DataFrame()
df_mic_base["Fecha"] = dt_mic.dt.normalize()          # solo fecha
df_mic_base["Hora"] = dt_mic.dt.strftime("%H:%M")     # solo hora (HH:MM)

# Clasificación DIURNA / NOCTURNA según la hora (1 a 5 am = NOCTURNA)
hora_h_num = dt_mic.dt.hour.fillna(0).astype(int)
df_mic_base["Hora_dia"] = np.where(
    hora_h_num.between(1, 5, inclusive="both"),
    "NOCTURNA",
    "DIURNA",
)

# --------------------------------------------------------------------
# 2) Columnas base para agrupar
# --------------------------------------------------------------------
df_mic_base["Empresa Transporte"] = df_mic_raw["Empresa_Transporte"].astype(str).str.strip()
df_mic_base["Linea"] = df_mic_raw["Linea"].astype(str).str.strip()
df_mic_base["Contrato"] = (
    df_mic_raw["ct_nombre"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
    .str.upper()
)

# Constantes para Micronauta
df_mic_base["Empresa Recaudadora"] = "Micronauta"
df_mic_base["Forma Pago"] = "Micronauta"

# Cada fila representa 1 boleto
df_mic_base["Boletos"] = 1

# --------------------------------------------------------------------
# 3) Agrupación Micronauta:
#    - Por Fecha, Empresa Transporte, Línea, Contrato, Hora_dia
#    - Boletos = COUNT de filas agrupadas (suma de 1)
#    - Tarifa = 0, Recaudacion = 0
# --------------------------------------------------------------------
group_cols = ["Fecha", "Empresa Transporte", "Linea", "Contrato", "Hora_dia"]

df_mic = (
    df_mic_base
    .groupby(group_cols, as_index=False)
    .agg({
        "Boletos": "sum",
        "Hora": "first",   # una hora de referencia dentro del grupo (opcional)
    })
)

df_mic["Boletos"] = df_mic["Boletos"].astype(int)
df_mic["Tarifa"] = 0.0
df_mic["Recaudacion"] = 0.0
df_mic["Empresa Recaudadora"] = "Micronauta"
df_mic["Forma Pago"] = "Micronauta"

# --------------------------------------------------------------------
# 4) Aplicar DIMENSIONES (igual que en SUBE, sobre el df_mic ya agrupado)
# --------------------------------------------------------------------
dim_contratos = dim_contratos_tipo.copy()
dim_juris = dim_jurisdicciones_tipo.copy()
dim_dias_merge = (
    dim_dias_tipo[["Día", "Tipo de día", "Nombre de día"]]
    .drop_duplicates("Día")
    .copy()
)
dim_tarifas = dim_tarifa_tipo.copy()

dim_dias_merge["Día"] = pd.to_datetime(dim_dias_merge["Día"]).dt.normalize()

# Contratos: CONTRATO / Contrato, normalizados igual que Micronauta
col_contrato_dim = "CONTRATO" if "CONTRATO" in dim_contratos.columns else "Contrato"
dim_contratos[col_contrato_dim] = (
    dim_contratos[col_contrato_dim]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
    .str.upper()
)

# TIPO por contrato
df_mic = df_mic.merge(
    dim_contratos[[col_contrato_dim, "TIPO"]].drop_duplicates(col_contrato_dim),
    left_on="Contrato",
    right_on=col_contrato_dim,
    how="left",
)
if col_contrato_dim in df_mic.columns:
    df_mic = df_mic.drop(columns=[col_contrato_dim])

# Asegurar que TIPO exista aunque el merge no traiga nada
if "TIPO" not in df_mic.columns:
    df_mic["TIPO"] = ""
df_mic["TIPO"] = df_mic["TIPO"].astype(str).str.upper().str.strip()

# Jurisdicción: normalizar Tipo contrato y TIPO en mayúsculas
dim_juris["Tipo contrato"] = (
    dim_juris["Tipo contrato"].astype(str).str.upper().str.strip()
)
df_mic = df_mic.merge(
    dim_juris[["Tipo contrato", "Jurisdicción"]].drop_duplicates("Tipo contrato"),
    left_on="TIPO",
    right_on="Tipo contrato",
    how="left",
).drop(columns=["Tipo contrato"])

# Tipo de día / Nombre de día (por fecha, igual que SUBE)
df_mic = df_mic.merge(
    dim_dias_merge,
    left_on="Fecha",
    right_on="Día",
    how="left",
).drop(columns=["Día"])

# ---- Tarifa2 por línea desde dim_tarifa_tipo ----
dim_tarifas.columns = [c.strip() for c in dim_tarifas.columns]
col_linea_tarifa = "Línea" if "Línea" in dim_tarifas.columns else "Linea"
col_tarifa2_dim = next(
    (c for c in dim_tarifas.columns if c.strip().upper() == "TARIFA2"),
    None,
)

if col_tarifa2_dim is not None:
    dim_tarifas[col_linea_tarifa] = (
        dim_tarifas[col_linea_tarifa]
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
    )

    df_mic["Linea"] = (
        df_mic["Linea"]
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
    )

    df_mic = df_mic.merge(
        dim_tarifas[[col_linea_tarifa, col_tarifa2_dim]].drop_duplicates(col_linea_tarifa),
        left_on="Linea",
        right_on=col_linea_tarifa,
        how="left",
    ).drop(columns=[col_linea_tarifa])

    if col_tarifa2_dim != "Tarifa2":
        df_mic = df_mic.rename(columns={col_tarifa2_dim: "Tarifa2"})

# Si después de todo no existe Tarifa2, crearla en 0 para evitar KeyError
if "Tarifa2" not in df_mic.columns:
    df_mic["Tarifa2"] = 0.0

# Ajuste nocturno en Tarifa2 según Tipo de día (igual que SUBE)
if "Tipo de día" in df_mic.columns:
    mask_nocturno_mic = (
        df_mic["Tipo de día"]
        .astype(str)
        .str.upper()
        .str.contains("NOCTURNO")
    )
    df_mic.loc[mask_nocturno_mic, "Tarifa2"] = (
        df_mic.loc[mask_nocturno_mic, "Tarifa2"] * 1.15
    ).round(2)

# --------------------------------------------------------------------
# 5) COLUMNAS CALCULADAS para Micronauta (igual que SUBE, pero Recaudacion ya es 0)
# --------------------------------------------------------------------
comision_por_contrato = (
    dim_contratos
    .drop_duplicates(col_contrato_dim)
    .set_index(col_contrato_dim)["COMISION"]
)
df_mic["COMISION"] = (
    -1 * df_mic["Recaudacion"] * df_mic["Contrato"].map(comision_por_contrato)
).round(2)

df_mic["Reca REAL"] = np.where(
    (df_mic["TIPO"] == "BOS") & (df_mic["Empresa Recaudadora"].str.upper() == "BIZLAND"),
    (df_mic["Recaudacion"] * 0.5).round(2),
    np.where(df_mic["TIPO"] == "BAM", 0, df_mic["Recaudacion"].round(2)),
)

df_mic["PERCIBIDO"] = np.where(
    df_mic["TIPO"].isin(["BAM", "BSC", "BEC"]),
    0,
    np.where(
        (df_mic["TIPO"] == "BOS") & (df_mic["Empresa Recaudadora"].str.upper() == "BIZLAND"),
        (df_mic["Recaudacion"] * 0.5 + df_mic["COMISION"]).round(2),
        (df_mic["Recaudacion"] + df_mic["COMISION"]).round(2),
    ),
)

df_mic["Mes"] = df_mic["Fecha"].dt.month
df_mic["Mes_nombre"] = df_mic["Fecha"].dt.month_name("es_ES")

# VALORIZACION con misma fórmula (usa Tarifa2 y Boletos)
forma_pago_integ_mic = df_mic["Forma Pago"].isin([
    "Uso Transporte con Integracion",
    "Uso Tarjeta Digital con Integración",
    "Uso tarjeta digital QR con integración",
])

cond_tarifa2_mic = (
    (df_mic["Jurisdicción"] == "MUNICIPIO") |
    (df_mic["TIPO"].isin(["BEC", "BSC", "BAM"]))
)

emp_rec_upper_mic = df_mic["Empresa Recaudadora"].astype(str).str.upper()

cond_atributos_mic = (df_mic["TIPO"] == "ATRIBUTOS")
cond_bos_micronauta_mic = (df_mic["TIPO"] == "BOS") & (emp_rec_upper_mic == "MICRONAUTA")
cond_bos_sube_mic = (df_mic["TIPO"] == "BOS") & (emp_rec_upper_mic == "SUBE")

valor_unitario_mic = np.select(
    [
        forma_pago_integ_mic,
        cond_tarifa2_mic,
        cond_atributos_mic,
        cond_bos_micronauta_mic,
        cond_bos_sube_mic,
    ],
    [
        0,
        df_mic["Tarifa2"],
        df_mic["Tarifa"] / 0.45 / 1.105 * 0.55,
        df_mic["Tarifa"] * 0.5,
        df_mic["Tarifa"],
    ],
    default=0,
)

df_mic["VALORIZACION"] = (valor_unitario_mic * df_mic["Boletos"]).round(2)

# --------------------------------------------------------------------
# 6) Unir SUBE + Micronauta en el mismo df_all_sube
# --------------------------------------------------------------------
df_all_sube = df_all_sube[
    df_all_sube["Empresa Recaudadora"].astype(str).str.upper().str.strip() != "MICRONAUTA"
].copy()

for col in df_all_sube.columns:
    if col not in df_mic.columns:
        df_mic[col] = np.nan

df_mic = df_mic[df_all_sube.columns]

df_all_sube = pd.concat([df_all_sube, df_mic], ignore_index=True)

df_mic_only = df_all_sube[
    df_all_sube["Empresa Recaudadora"].astype(str).str.upper().str.strip() == "MICRONAUTA"
].copy()
print(sorted(df_mic_only["Fecha"].dt.to_period("M").dropna().unique().astype(str)))
print(df_mic_only["Fecha"].max())
print("Filas totales combinadas (SUBE + Micronauta):", len(df_all_sube))

Columnas en query micronauta (limpias): ['transad_id', 'Fecha', 'Empresa_Recaudadora', 'Forma_Pago', 'Empresa_Transporte', 'Linea', 'ct_nombre', 'Tarifa', 'Cant_Boletos', 'Recaudacion']
['2026-01', '2026-02', '2026-03']
2026-03-06 00:00:00
Filas totales combinadas (SUBE + Micronauta): 338443


In [6]:
df_mic_only = df_all_sube[df_all_sube["Empresa Recaudadora"] == "Micronauta"].copy()
print(
    sorted(df_mic_only["Fecha"].dt.to_period("M").dropna().unique().astype(str))
)
print(df_mic_only["Fecha"].max())

['2026-01', '2026-02', '2026-03']
2026-03-06 00:00:00


### Resumen Mensualt y Total

In [7]:
# Resumen del MES ANTERIOR al vigente real
# ========================================

# Asegurar que Fecha es datetime
df_all_sube["Fecha"] = pd.to_datetime(df_all_sube["Fecha"], errors="coerce")

# Empresas a mostrar en columnas
empresas = ["Coniferal", "Grupo FAM Urbano", "SiBus CBA", "Tamsau"]

# Fecha de hoy
hoy = pd.Timestamp.today().normalize()

# Primer día del mes actual
primer_dia_mes_actual = hoy.replace(day=1)

# Mes anterior al de hoy (ej: si hoy es marzo, esto da 1/feb)
fecha_mes_anterior = primer_dia_mes_actual - pd.DateOffset(months=1)
mes_vigente = fecha_mes_anterior.month
anio_vigente = fecha_mes_anterior.year
nombre_mes_vigente = fecha_mes_anterior.month_name("es_ES").capitalize()

print(f"\nMes tomado para el resumen mensual: {nombre_mes_vigente} {anio_vigente}")

# Filtrar df_all_sube SOLO al mes/año seleccionado (mes anterior)
df_mes = df_all_sube[
    (df_all_sube["Fecha"].dt.month == mes_vigente) &
    (df_all_sube["Fecha"].dt.year == anio_vigente)
].copy()

print("Filas en df_mes (mes seleccionado):", len(df_mes))

totales_mes = (
    df_mes
    .groupby("Empresa Transporte")[["Reca REAL", "PERCIBIDO", "VALORIZACION"]]
    .sum()
)

resumen_mes = pd.DataFrame(
    index=["Suma de Reca REAL", "Suma de PERCIBIDO", "Suma de VALORIZACIÓN"],
    columns=empresas,
    dtype=float,
)

for emp in empresas:
    if emp in totales_mes.index:
        resumen_mes.loc["Suma de Reca REAL", emp] = totales_mes.loc[emp, "Reca REAL"]
        resumen_mes.loc["Suma de PERCIBIDO", emp] = totales_mes.loc[emp, "PERCIBIDO"]
        resumen_mes.loc["Suma de VALORIZACIÓN", emp] = totales_mes.loc[emp, "VALORIZACION"]
    else:
        resumen_mes[emp] = 0.0

resumen_mes["Total general"] = resumen_mes[empresas].sum(axis=1)
resumen_mes = resumen_mes.fillna(0).round(2)

print(f"\nRESUMEN MENSUAL - {nombre_mes_vigente} {anio_vigente}")
display(resumen_mes)

df_resumen_mes = resumen_mes.reset_index().rename(columns={"index": "Valores"})


Mes tomado para el resumen mensual: Febrero 2026
Filas en df_mes (mes seleccionado): 81605

RESUMEN MENSUAL - Febrero 2026


,Coniferal,Grupo FAM Urbano,SiBus CBA,Tamsau,Total general
Suma de Reca REAL,3.172455e+09,1.972848e+09,1.369135e+09,5.906376e+08,7.105076e+09
Suma de PERCIBIDO,3.017039e+09,1.876201e+09,1.302131e+09,5.619953e+08,6.757367e+09
Suma de VALORIZACIÓN,1.439386e+09,9.791065e+08,6.772473e+08,2.779942e+08,3.373734e+09


### hoja Liquidacion

In [8]:
# 3) Hoja "Liquidaciones" -> archivo "liquid2"
liq_path = os.path.join(BASE, "Referencias de consolidados Sube.xlsx")

df_liquidaciones = pd.read_excel(liq_path)

# Limpiar posibles BOM y espacios
df_liquidaciones.columns = [c.replace("ï»¿", "").strip() for c in df_liquidaciones.columns]

print("Columnas finales en liquid2:", df_liquidaciones.columns.tolist())
# Debería mostrar: ['Liquidacion', 'Fecha'] (al menos)

# Fecha derivada del nombre de liquidación:
# Ejemplo: 633_20250512_124734_3_RespaldoLiquidacionProc.csv
#           012345678
#                ^--- desde índice 4, 8 caracteres -> '20250512'
liquid_str = df_liquidaciones["Liquidacion"].astype(str)
fecha_str8 = liquid_str.str[4:12]   # posiciones 4 a 11 (0-based) -> 'YYYYMMDD'

df_liquidaciones["Fecha"] = pd.to_datetime(
    fecha_str8,
    format="%Y%m%d",
    errors="coerce"
)

# Código empresa = primeros 3 caracteres de Liquidacion
cod_liq = liquid_str.str[:3]

mapeo_empresas_liq = {
    "633": "Coniferal",
    "634": "Tamsau",
    "636": "SiBus CBA",
    "637": "Grupo FAM Urbano",
}

df_liquidaciones["Empresa Transporte"] = cod_liq.map(mapeo_empresas_liq).fillna("Otro")

Columnas finales en liquid2: ['Liquidacion']


In [9]:
# ==================================================
# Liquidación por mes — un CSV a la vez (solo FECHA DE TRX), bajo uso de RAM
# ==================================================

BASE_LIQ = BASE if "BASE" in dir() else r"C:\Users\usuario\OneDrive\Desktop\Reporte"
SUBE_HIST_LIQ = Path(BASE_LIQ) / "Sube_historica"

mapeo_empresas_liq = {
    "633": "Coniferal",
    "634": "Tamsau",
    "636": "SiBus CBA",
    "637": "Grupo FAM Urbano",
}

orden_meses = [5, 6, 7, 8, 9, 10, 11, 12, 1, 2, 3]
map_mes_col = {
    5:  "Cantidad Mayo",
    6:  "Cantidad Junio",
    7:  "Cantidad Julio",
    8:  "Cantidad Agosto",
    9:  "Cantidad Septiembre",
    10: "Cantidad Octubre",
    11: "Cantidad Noviembre",
    12: "Cantidad Diciembre",
    1:  "Cantidad Enero",
    2:  "Cantidad Febrero",
    3:  "Cantidad Marzo",
}

rows_liq: list[dict] = []

if not SUBE_HIST_LIQ.is_dir():
    print(f"Carpeta no encontrada: {SUBE_HIST_LIQ}")
    df_liquidacion_mes = pd.DataFrame()
else:
    archivos_liq = sorted(SUBE_HIST_LIQ.glob("*.csv"))
    n_liq = len(archivos_liq)
    for idx, f in enumerate(archivos_liq, 1):
        nombre = f.name
        ruta_full = str(f.resolve())
        fecha_archivo_str8 = nombre[4:12] if len(nombre) >= 12 else ""
        fecha_archivo = pd.to_datetime(
            fecha_archivo_str8, format="%Y%m%d", errors="coerce"
        )
        cod_emp = nombre[:3]
        emp_tr = mapeo_empresas_liq.get(cod_emp, "Otro")

        try:
            s = pd.read_csv(
                f,
                sep=";",
                encoding="latin1",
                usecols=["FECHA DE TRX"],
                dtype=str,
                low_memory=False,
            )
        except (ValueError, KeyError, pd.errors.ParserError):
            s = pd.read_csv(f, sep=";", encoding="latin1", dtype=str, low_memory=False)
            if "FECHA DE TRX" not in s.columns:
                continue
            s = s[["FECHA DE TRX"]]

        fechas = pd.to_datetime(s.iloc[:, 0], dayfirst=True, errors="coerce")
        mes = fechas.dt.month
        vc = mes.value_counts(dropna=True)

        fila = {
            "Empresa": "Sube",
            "archivo_csv": ruta_full,
            "Fecha_archivo": fecha_archivo,
            "Empresa Transporte": emp_tr,
        }
        for m, col in map_mes_col.items():
            fila[col] = int(vc.get(m, 0) or 0)
        rows_liq.append(fila)

        if idx % 80 == 0 or idx == n_liq:
            print(f"  Liquidación por mes: {idx}/{n_liq} archivos")

    df_liquidacion_mes = pd.DataFrame(rows_liq)
    cols_orden = (
        ["Empresa", "archivo_csv", "Fecha_archivo", "Empresa Transporte"]
        + [map_mes_col[m] for m in orden_meses]
    )
    df_liquidacion_mes = (
        df_liquidacion_mes[cols_orden]
        .sort_values(["Fecha_archivo", "Empresa", "archivo_csv"])
        .rename(columns={"archivo_csv": "Liquidación", "Fecha_archivo": "Fecha"})
    )

df_liquidacion_mes.head() if len(df_liquidacion_mes) else None

### Exportacion a EXCEL

In [10]:
# Exportar reporte SUBE a Excel con todas las hojas
# ================================================

OUTPUT_EXCEL_SUBE = os.path.join(BASE, "reporte_sube.xlsx")

# Aseguramos que Fecha sea datetime
df_all_sube["Fecha"] = pd.to_datetime(df_all_sube["Fecha"], errors="coerce")

# Empresas en el orden deseado
empresas = ["Coniferal", "Grupo FAM Urbano", "SiBus CBA", "Tamsau"]

# -------- RESUMEN: SOLO MES ANTERIOR AL VIGENTE --------

# Fecha de hoy
hoy = pd.Timestamp.today().normalize()

# Primer día del mes actual
primer_dia_mes_actual = hoy.replace(day=1)

# Mes anterior al de hoy (ej: si hoy es marzo, esto da 1/feb)
fecha_mes_anterior = primer_dia_mes_actual - pd.DateOffset(months=1)
mes_vigente = fecha_mes_anterior.month
anio_vigente = fecha_mes_anterior.year
nombre_mes_vigente = fecha_mes_anterior.month_name("es_ES").capitalize()

# Filtrar df_all_sube SOLO al mes/año seleccionado (mes anterior)
df_mes = df_all_sube[
    (df_all_sube["Fecha"].dt.month == mes_vigente) &
    (df_all_sube["Fecha"].dt.year == anio_vigente)
].copy()

# Si no hubiera datos del mes anterior, evitamos errores
if df_mes.empty:
    # tabla vacía con la estructura esperada
    df_resumen_total = pd.DataFrame({
        "Valores": [
            "Suma de Reca REAL",
            "Suma de PERCIBIDO",
            "Suma de VALORIZACIÓN",
        ]
    })
    for emp in empresas:
        df_resumen_total[emp] = [0.0, 0.0, 0.0]
    df_resumen_total["Total general"] = df_resumen_total[empresas].sum(axis=1)

    titulo_mes = f"Valores - {nombre_mes_vigente} {anio_vigente}"
    df_resumen_mes = df_resumen_total.copy()
else:
    # Agrupar SOLO el mes anterior por Empresa Transporte
    resumen_emp_mes = (
        df_mes
        .groupby("Empresa Transporte")[["Reca REAL", "PERCIBIDO", "VALORIZACION"]]
        .sum()
        .reindex(empresas)
        .fillna(0)
    )

    df_resumen_total = pd.DataFrame({
        "Valores": [
            "Suma de Reca REAL",
            "Suma de PERCIBIDO",
            "Suma de VALORIZACIÓN",
        ]
    })

    for emp in empresas:
        df_resumen_total[emp] = [
            resumen_emp_mes.loc[emp, "Reca REAL"],
            resumen_emp_mes.loc[emp, "PERCIBIDO"],
            resumen_emp_mes.loc[emp, "VALORIZACION"],
        ]

    df_resumen_total["Total general"] = df_resumen_total[empresas].sum(axis=1)

    titulo_mes = f"Valores - {nombre_mes_vigente} {anio_vigente}"
    # Usamos la misma tabla para el bloque de “mes vigente” en la hoja
    df_resumen_mes = df_resumen_total.copy()

# -------- Agregar boletos para exportar (evitar límite Excel 1.048.576 filas) --------
group_cols = ["Fecha", "Empresa Recaudadora", "Empresa Transporte", "Forma Pago", "Linea", "Contrato", "Tarifa"]
sum_cols = ["Boletos", "Recaudacion", "COMISION", "Reca REAL", "PERCIBIDO", "VALORIZACION"]
other_cols = [c for c in df_all_sube.columns if c not in group_cols and c not in sum_cols]
agg_dict = {c: "sum" for c in sum_cols if c in df_all_sube.columns}
for c in other_cols:
    agg_dict[c] = "first"

df_export_boletos = df_all_sube.groupby(group_cols, as_index=False).agg(agg_dict)

# -------- Exportar todo --------
with pd.ExcelWriter(OUTPUT_EXCEL_SUBE, engine="xlsxwriter") as writer:
    # Boletos agregados (group by para no superar límite de filas de Excel)
    df_export_boletos.to_excel(writer, sheet_name="Boletos", index=False)

    # Hoja de resúmenes
    hoja_resumen = "LIQUIDACIÓN II OPC A"

    # Resumen TOTAL (pero SOLO del mes anterior)
    df_resumen_total.to_excel(
        writer,
        sheet_name=hoja_resumen,
        index=False,
        startrow=0,
        startcol=0,
    )

    # Dos filas vacías
    startrow_mes = len(df_resumen_total) + 3

    # Escribir título del mes anterior en una fila sola
    workbook  = writer.book
    worksheet = writer.sheets[hoja_resumen]
    worksheet.write(startrow_mes, 0, titulo_mes)

    # Resumen del mes anterior, empezando dos filas debajo del título
    df_resumen_mes.to_excel(
        writer,
        sheet_name=hoja_resumen,
        index=False,
        startrow=startrow_mes + 2,
        startcol=0,
    )

    # Hoja "Km por línea"
    df_km_por_linea.to_excel(
        writer,
        sheet_name="Km por línea",
        index=False,
    )

    # Hoja "Km entre paradas"
    df_km_entre_paradas.to_excel(
        writer,
        sheet_name="Km entre paradas",
        index=False,
    )

    # Hoja "Liquidaciones"
    df_liquidaciones.to_excel(
        writer,
        sheet_name="Liquidaciones",
        index=False,
    )
    
    df_liquidacion_mes.to_excel(
        writer,
        sheet_name="Liquidacion por mes",
        index=False,
    )

print("Archivo Excel generado:", OUTPUT_EXCEL_SUBE)

Archivo Excel generado: C:\Users\usuario\OneDrive\Desktop\Reporte\reporte_sube.xlsx
